# 🧳 Building a Travel Booking Agent

This lab builds an agent that plans a trip end to end:

1. **Asks** who is travelling — how many adults and children, and from where
2. **Checks** whether you already have a similar booking
3. **Finds flights** using `$graphLookup` graph traversal over real airline routes
4. **Suggests 4–5 stays** using vector search filtered by party size and child amenities
5. **Pauses for your approval** before writing anything
6. **Creates a booking draft** only once you agree

Along the way you will use MongoDB as a **vector database**, a **graph database**, and an
**operational database** — plus LangGraph for the agent loop, human-in-the-loop approval,
and memory.

# Step 1: Setup prerequisites

- Set the LLM provider and passkey provided by your workshop instructor

- `LLM_PROVIDER` can be set to one of "aws"/ "microsoft" / "google"

In [ ]:
LLM_PROVIDER = "aws"
PASSKEY = "replace-with-passkey"

In [ ]:
import os
import sys
from pymongo import MongoClient

# Add parent directory to path to import from utils
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from utils import set_env

# ----- MONGODB SETUP -----
# If you are using your own MongoDB Atlas cluster, use the connection string for your cluster here
MONGODB_URI = os.environ.get("MONGODB_URI")
# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI, appname="devrel-workshop-travel-agent")
# Check the connection to the server
mongodb_client.admin.command("ping")

# ----- API KEY SETUP -----
# Obtain API keys from our AI model proxy and set them as environment variables-- DO NOT CHANGE
set_env([LLM_PROVIDER, "voyageai"], PASSKEY)

# Step 2: Connect to the travel datasets

| Collection | Source | Contents |
| --- | --- | --- |
| `airbnb_listings_embeddings` | `sample_airbnb` + Voyage AI | 5,555 stays with 1024-dim embeddings |
| `airbnb_listings` | `sample_airbnb` | the same stays, full detail |
| `sample_training.routes` | Atlas sample data | 66,985 real airline routes |

The stays were pre-embedded with **`voyage-4-large`** so you don't have to wait for
5,555 API calls. The routes come from the Atlas sample dataset already on your cluster.

### **Do not change the values assigned to the variables below**

In [ ]:
# Database holding the prepared travel data
DB_NAME = "mongodb_genai_devday_travel_agent"
# Stays with embeddings- used for semantic search
VS_COLLECTION_NAME = "airbnb_listings_embeddings"
# Full stay documents- used for detail lookups
FULL_COLLECTION_NAME = "airbnb_listings"
# Booking drafts created by the agent
DRAFTS_COLLECTION_NAME = "booking_drafts"
# Name of the vector search index
VS_INDEX_NAME = "vector_index"

# Atlas sample dataset holding real airline routes
ROUTES_DB_NAME = "sample_training"
ROUTES_COLLECTION_NAME = "routes"

In [ ]:
# Stays for vector search
vs_collection = mongodb_client[DB_NAME][VS_COLLECTION_NAME]
# Full stay documents
full_collection = mongodb_client[DB_NAME][FULL_COLLECTION_NAME]
# Booking drafts written by the agent
drafts_collection = mongodb_client[DB_NAME][DRAFTS_COLLECTION_NAME]
# Airline routes- the graph we will traverse
routes_collection = mongodb_client[ROUTES_DB_NAME][ROUTES_COLLECTION_NAME]

print(f"{vs_collection.count_documents({}):,} stays with embeddings")
print(f"{routes_collection.count_documents({}):,} airline routes")

# Step 3: Create a vector search index with filters

In [ ]:
from utils import create_search_index, check_index_ready, create_index

A travel agent can't rely on semantics alone. "A place in Barcelona for 4 people"
has a **semantic** part (*what the place is like*) and two **hard constraints**
(*city* and *capacity*) that must never be violated — a romantic studio for 2 is
the wrong answer no matter how well it matches the description.

Declaring `filter` fields lets Atlas apply those constraints **during** the vector
search rather than discarding results afterwards, so you still get a full set of
matches back.

📚 https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-type/#about-the-filter-type

In [ ]:
# Vector index definition with two filter fields alongside the vector field:
# path: Path to the embeddings field
# numDimensions: 1024- the output dimension of voyage-4-large
# similarity: Similarity metric. One of cosine, euclidean, dotProduct.
model = {
    "name": VS_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine",
            },
            # Filter fields- used to enforce hard constraints during the search
            <CODE_BLOCK_1>
        ]
    },
}

In [ ]:
# Create the index, then wait until it is READY before querying it
create_search_index(vs_collection, VS_INDEX_NAME, model)
check_index_ready(vs_collection, VS_INDEX_NAME)

# Step 4: Flight search with `$graphLookup`

Flight routes are a **graph**: airports are nodes and each route document is an edge.
Finding a trip from A to B is a graph traversal, and MongoDB does this natively with
`$graphLookup` — no separate graph database required.

Each route document looks like this:

```json
{ "airline": { "name": "TAP Portugal", "iata": "TAP" },
  "src_airport": "BCN", "dst_airport": "OPO",
  "airplane": "319", "stops": 0 }
```

📚 https://www.mongodb.com/resources/basics/databases/mongodb-graph-database

In [ ]:
import time
from pprint import pprint
from typing import List

# The agent talks in city names; the routes dataset uses IATA airport codes.
CITY_AIRPORTS = {
    "new york": ["JFK", "LGA", "EWR"],
    "barcelona": ["BCN"],
    "porto": ["OPO"],
    "istanbul": ["IST", "SAW"],
    "hong kong": ["HKG"],
    "sydney": ["SYD"],
    "montreal": ["YUL"],
    "rio de janeiro": ["GIG", "SDU"],
    "oahu": ["HNL"],
    "maui": ["OGG"],
    "kauai": ["LIH"],
    "the big island": ["KOA"],
}


def airports_for(city: str) -> List[str]:
    """Look up IATA airport codes for a city name."""
    return CITY_AIRPORTS.get(city.strip().lower(), [])

### 4a. Why indexing matters for graph traversal

`$graphLookup` re-queries `connectToField` **at every hop**. Without an index on that
field every hop becomes a full collection scan of ~67,000 documents — and a hub
airport like JFK (456 outbound routes) multiplies that cost by 456.

Let's measure it. The Atlas sample dataset ships with only the default `_id` index:

In [ ]:
print("indexes on routes:", [ix["name"] for ix in routes_collection.list_indexes()])

In [ ]:
# Find one-stop itineraries from JFK to Porto (OPO).
# `restrictSearchWithMatch` keeps only traversed routes that land at the destination.
connection_pipeline = [
    {"$match": {"src_airport": "JFK"}},
    {
        "$graphLookup": {
            "from": ROUTES_COLLECTION_NAME,
            <CODE_BLOCK_2>
            "as": "connections",
            "maxDepth": 0,
            "depthField": "depth",
            <CODE_BLOCK_3>
        }
    },
    {"$match": {"connections.0": {"$exists": True}}},
    {"$count": "paths"},
]

start = time.time()
result = list(routes_collection.aggregate(connection_pipeline))
print(f"JFK -> OPO: {result[0]['paths']} paths in {time.time() - start:.2f}s")

That took **several seconds** — far too slow for a tool the agent calls repeatedly.

Now index the two fields the traversal connects on:

In [ ]:
# `src_airport` is the connectToField- indexing it is what makes traversal fast
<CODE_BLOCK_4>
create_index(routes_collection, [("dst_airport", 1)], "dst_airport_1")

In [ ]:
# Re-run the IDENTICAL query and compare
start = time.time()
result = list(routes_collection.aggregate(connection_pipeline))
print(f"JFK -> OPO: {result[0]['paths']} paths in {time.time() - start:.2f}s")

**~9s → ~0.2s: about a 40x speedup** on the same query returning the same results.

Confirm *why* by inspecting the query plan — you should now see `IXSCAN` (index scan)
where there was previously `COLLSCAN` (collection scan):

In [ ]:
from bson import json_util

explain = mongodb_client[ROUTES_DB_NAME].command(
    "explain",
    {
        "aggregate": ROUTES_COLLECTION_NAME,
        # Drop the $count stage so the planner reports the scan strategy
        "pipeline": connection_pipeline[:-1],
        "cursor": {},
    },
    verbosity="queryPlanner",
)
plan = json_util.dumps(explain)
print("IXSCAN   occurrences:", plan.count("IXSCAN"))
print("COLLSCAN occurrences:", plan.count("COLLSCAN"))

# Step 5: Create the agent tools

Five tools, each doing one job:

| Tool | What it does |
| --- | --- |
| `get_trip_requirements` | reports which details are still missing |
| `check_existing_bookings` | avoids creating a duplicate booking |
| `search_flights` | `$graphLookup` traversal: direct, else one stop |
| `suggest_stays` | filtered vector search for 4–5 candidate stays |
| `create_booking_draft` | writes the draft — only after you approve |

In [ ]:
from langchain.agents import tool
from typing import Dict, Optional
import voyageai

# Initialize the Voyage AI client
vo = voyageai.Client()

### Embedding the query

The 5,555 stays were embedded once with **`voyage-4-large`** (the highest-quality model).
Queries are embedded with **`voyage-4`** instead — it is faster and cheaper, and every
model in the Voyage 4 series shares the same vector space, so the two are directly
comparable.

That asymmetry is deliberate and it is how production search systems are built: you
embed the catalogue **once**, but you embed a query on **every single search**.

📚 https://docs.voyageai.com/docs/embeddings

In [ ]:
# Model used to embed the 5,555 stays (already done for you)
DOC_MODEL = "voyage-4-large"
# Model used to embed user queries at runtime
QUERY_MODEL = "voyage-4"


def get_embeddings(query: str, model: str = QUERY_MODEL) -> List[float]:
    """
    Get embeddings for an input query.

    Args:
        query (str): Query string
        model (str): Voyage model to use

    Returns:
        List[float]: Embedding of the query string
    """
    # Use `input_type="query"` so Voyage optimizes the vector for retrieval
    <CODE_BLOCK_5>

In [ ]:
# One index, three interchangeable query models- try it yourself
for candidate in ["voyage-4-large", "voyage-4", "voyage-4-lite"]:
    start = time.time()
    embedding = get_embeddings("quiet apartment near the beach", candidate)
    print(f"{candidate:16s} dims={len(embedding)}  {(time.time() - start) * 1000:.0f}ms")

### Tool 1: Search flights

Try **direct** routes first. If there are none, fall back to `$graphLookup` to find
**one-stop** itineraries. Porto is a good example: there is no direct JFK–OPO route,
but you can connect through Barcelona, Paris, Dublin or Rome.

> **Note:** the routes are real, but this dataset has no schedules, seat counts or
> fares. Prices and seat availability below are **derived deterministically from the
> route** so the lab behaves consistently — they are illustrative, not real inventory.

In [ ]:
def estimate_fare(src: str, dst: str, stops: int) -> int:
    """Deterministic pseudo-fare so the lab gives repeatable results."""
    base = 180 + (sum(ord(ch) for ch in src + dst) % 420)
    return base + (120 * stops)


def estimate_seats(src: str, dst: str) -> int:
    """Deterministic pseudo seat count between 0 and 8."""
    return sum(ord(ch) for ch in dst + src) % 9

In [ ]:
@tool
def search_flights(origin_city: str, destination_city: str, travellers: int = 1) -> str:
    """
    Find flights between two cities for a given number of travellers.
    Returns direct flights when they exist, otherwise one-stop connections.

    Args:
        origin_city (str): City the traveller departs from, e.g. "New York".
        destination_city (str): City the traveller is flying to, e.g. "Barcelona".
        travellers (int): Total number of travellers, including children.

    Returns:
        str: Human readable list of flight options.
    """
    origins = airports_for(origin_city)
    destinations = airports_for(destination_city)
    if not origins:
        return f"I don't have flight data for {origin_city}."
    if not destinations:
        return f"I don't have flight data for {destination_city}."

    # ---- 1. Direct routes ----
    direct = list(
        routes_collection.aggregate(
            [
                {
                    "$match": {
                        "src_airport": {"$in": origins},
                        "dst_airport": {"$in": destinations},
                    }
                },
                {
                    "$group": {
                        "_id": {
                            "airline": "$airline.name",
                            "src": "$src_airport",
                            "dst": "$dst_airport",
                        }
                    }
                },
                {"$limit": 6},
            ]
        )
    )

    options = []
    for row in direct:
        src, dst = row["_id"]["src"], row["_id"]["dst"]
        seats = estimate_seats(src, dst)
        if seats < travellers:
            continue  # not enough seats for the whole party
        fare = estimate_fare(src, dst, 0)
        options.append(
            f"DIRECT | {row['_id']['airline']} | {src} -> {dst} | "
            f"${fare} per person | ${fare * travellers} total | {seats} seats left"
        )

    if options:
        return "Direct flights found:\n" + "\n".join(options)

    # ---- 2. No direct route: traverse the graph for one-stop options ----
    connecting = list(
        routes_collection.aggregate(
            [
                {"$match": {"src_airport": {"$in": origins}}},
                {
                    "$graphLookup": {
                        "from": ROUTES_COLLECTION_NAME,
                        <CODE_BLOCK_6>
                        "as": "second_leg",
                        "maxDepth": 0,
                        <CODE_BLOCK_7>
                    }
                },
                {"$match": {"second_leg.0": {"$exists": True}}},
                {
                    "$project": {
                        "_id": 0,
                        "leg1_airline": "$airline.name",
                        "src": "$src_airport",
                        "hub": "$dst_airport",
                        "second_leg": {"$slice": ["$second_leg", 1]},
                    }
                },
                {"$limit": 5},
            ]
        )
    )

    if not connecting:
        return (
            f"No flights found from {origin_city} to {destination_city}, "
            "even with one stop. Consider a different destination."
        )

    for row in connecting:
        leg2 = row["second_leg"][0]
        fare = estimate_fare(row["src"], leg2["dst_airport"], 1)
        options.append(
            f"1 STOP | {row['leg1_airline']} then {leg2['airline']['name']} | "
            f"{row['src']} -> {row['hub']} -> {leg2['dst_airport']} | "
            f"${fare} per person | ${fare * travellers} total"
        )

    return (
        f"No direct flights from {origin_city} to {destination_city}. "
        "One-stop options:\n" + "\n".join(options)
    )

### Tool 2: Suggest stays

This is where the three techniques combine in a single query:

1. **`$vectorSearch`** — semantic match on what the traveller described
2. **`filter`** — hard constraints on city and capacity (the fields declared in Step 3)
3. **`$match`** — child amenities and availability, applied after retrieval

Children change the search in a real way: they raise the required capacity **and** make
cribs and high chairs relevant. `availability_365 > 0` drops the ~1,200 listings that
are fully booked.

In [ ]:
# Amenities that matter when travelling with young children
CHILD_AMENITIES = [
    "Family/kid friendly",
    "Crib",
    "Pack \u2019n Play/travel crib",
    "High chair",
    "Children\u2019s books and toys",
]


@tool
def suggest_stays(
    destination_city: str, adults: int, kids: int = 0, preferences: str = ""
) -> str:
    """
    Suggest places to stay in a city for a given party, using semantic search.

    Args:
        destination_city (str): City to search in, e.g. "Barcelona".
        adults (int): Number of adults.
        kids (int): Number of children.
        preferences (str): Free text describing what the traveller wants,
            e.g. "quiet, near the beach, with a kitchen".

    Returns:
        str: Numbered list of candidate stays.
    """
    party = adults + kids

    # Describe the trip in natural language, then embed that description
    query = f"place to stay in {destination_city} for {adults} adults"
    if kids:
        query += f" and {kids} children, child friendly"
    if preferences:
        query += f". {preferences}"
    query_embedding = get_embeddings(query)

    # Hard constraints applied DURING the vector search
    <CODE_BLOCK_8>

    pipeline = [
        {
            "$vectorSearch": {
                "index": VS_INDEX_NAME,
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 200,
                "limit": 25,
                <CODE_BLOCK_9>
            }
        },
        # Only keep listings that still have availability
        <CODE_BLOCK_10>
    ]

    # With children, require at least one child-specific amenity
    if kids:
        <CODE_BLOCK_11>

    pipeline += [
        {
            "$project": {
                "_id": 0,
                "listing_id": "$_id",
                "name": 1,
                "property_type": 1,
                "room_type": 1,
                "accommodates": 1,
                "bedrooms": 1,
                "price": 1,
                "rating": "$review_scores.review_scores_rating",
                "minimum_nights": 1,
                "score": {"$meta": "vectorSearchScore"},
            }
        },
        # Return 5 candidates for the traveller to choose from
        {"$limit": 5},
    ]

    results = list(vs_collection.aggregate(pipeline))
    if not results:
        return (
            f"No available stays in {destination_city} for {party} guests. "
            "Try a smaller party or a different city."
        )

    lines = [f"Found {len(results)} options in {destination_city} for {party} guests:"]
    for i, stay in enumerate(results, start=1):
        # NOTE: minimum_nights is stored as a STRING in this dataset
        min_nights = int(stay.get("minimum_nights") or 1)
        lines.append(
            f"{i}. {stay['name']} | {stay.get('property_type')} | "
            f"sleeps {stay['accommodates']} | ${float(stay['price']):g}/night | "
            f"rating {stay.get('rating') or 'n/a'}/100 | min {min_nights} nights "
            f"| id={stay['listing_id']}"
        )
    return "\n".join(lines)

### Tool 3: Check what's still missing

A booking needs a name, a party size, an origin, a destination and dates. This tool
lets the agent see what it still has to ask for, instead of guessing.

In [ ]:
# Everything required before a booking draft can be created
REQUIRED_SLOTS = [
    "full_name",
    "adults",
    "origin_city",
    "destination_city",
    "check_in",
    "check_out",
]


@tool
def get_trip_requirements(
    full_name: str = "",
    adults: int = 0,
    kids: int = 0,
    origin_city: str = "",
    destination_city: str = "",
    check_in: str = "",
    check_out: str = "",
) -> str:
    """
    Check which trip details are still missing before a booking can be drafted.
    Call this early to find out what to ask the traveller for.

    Args:
        full_name (str): Name the booking will be made under.
        adults (int): Number of adults.
        kids (int): Number of children.
        origin_city (str): City the traveller departs from.
        destination_city (str): City the traveller wants to visit.
        check_in (str): Check-in date as YYYY-MM-DD.
        check_out (str): Check-out date as YYYY-MM-DD.

    Returns:
        str: Which details are present and which are still needed.
    """
    provided = {
        "full_name": full_name,
        "adults": adults,
        "kids": kids,
        "origin_city": origin_city,
        "destination_city": destination_city,
        "check_in": check_in,
        "check_out": check_out,
    }
    missing = [slot for slot in REQUIRED_SLOTS if not provided.get(slot)]

    known = ", ".join(f"{k}={v}" for k, v in provided.items() if v) or "nothing yet"
    if missing:
        return (
            f"Known so far: {known}.\n"
            f"Still needed: {', '.join(missing)}.\n"
            "Ask the traveller for the missing details before searching."
        )
    return f"All required details collected: {known}. You can search now."

### Tool 4: Check for an existing booking

Before creating anything, look for a booking this traveller already has for the same
destination and overlapping dates. Two overlapping date ranges satisfy
`existing.check_in < new.check_out` **and** `existing.check_out > new.check_in`.

In [ ]:
@tool
def check_existing_bookings(
    full_name: str, destination_city: str, check_in: str, check_out: str
) -> str:
    """
    Check whether the traveller already has a booking that overlaps this trip.
    Always call this BEFORE creating a booking draft.

    Args:
        full_name (str): Name on the booking.
        destination_city (str): Destination city.
        check_in (str): Proposed check-in date as YYYY-MM-DD.
        check_out (str): Proposed check-out date as YYYY-MM-DD.

    Returns:
        str: Details of any overlapping booking, or confirmation that there is none.
    """
    existing = list(
        drafts_collection.find(
            {
                "full_name": full_name,
                "destination": destination_city,
                # Date ranges overlap
                <CODE_BLOCK_12>
            },
            {"_id": 1, "stay.name": 1, "booking_dates": 1, "estimated_total": 1},
        )
    )

    if not existing:
        return (
            f"No existing booking for {full_name} in {destination_city} "
            f"between {check_in} and {check_out}. Safe to continue."
        )

    lines = [f"{full_name} already has {len(existing)} overlapping booking(s):"]
    for row in existing:
        dates = row.get("booking_dates", {})
        lines.append(
            f"- {row['_id']}: {row.get('stay', {}).get('name', 'unknown stay')} "
            f"({dates.get('check_in')} to {dates.get('check_out')}, "
            f"${row.get('estimated_total', 0):g})"
        )
    lines.append("Tell the traveller about this instead of creating a duplicate.")
    return "\n".join(lines)

### Tool 5: Create the booking draft

The only tool that **writes**. It validates first and refuses to create anything that
doesn't add up:

- the stay must exist and still have availability
- the party must fit within `accommodates`
- the trip must be at least as long as `minimum_nights`

Note the `_id` is a readable slug built from name, city and date. Re-running the same
booking **updates** it rather than creating a duplicate — the write is idempotent.

In [ ]:
import re
from datetime import datetime, timezone


def slugify(*parts: str) -> str:
    """Build a readable, stable document id from the trip details."""
    raw = "_".join(str(p) for p in parts if p)
    return re.sub(r"[^a-z0-9]+", "_", raw.lower()).strip("_")


def nights_between(check_in: str, check_out: str) -> int:
    """Number of nights between two YYYY-MM-DD dates."""
    fmt = "%Y-%m-%d"
    return (datetime.strptime(check_out, fmt) - datetime.strptime(check_in, fmt)).days

In [ ]:
@tool
def create_booking_draft(
    full_name: str,
    listing_id: str,
    destination_city: str,
    check_in: str,
    check_out: str,
    adults: int,
    kids: int = 0,
    flight_summary: str = "",
    flight_price_total: float = 0.0,
) -> str:
    """
    Create a draft booking record. Only call this after the traveller has
    explicitly approved a specific stay and flight.

    Args:
        full_name (str): Name on the booking.
        listing_id (str): The `id=` value of the chosen stay from suggest_stays.
        destination_city (str): Destination city.
        check_in (str): Check-in date as YYYY-MM-DD.
        check_out (str): Check-out date as YYYY-MM-DD.
        adults (int): Number of adults.
        kids (int): Number of children.
        flight_summary (str): The chosen flight, as shown by search_flights.
        flight_price_total (float): Total flight cost for the whole party.

    Returns:
        str: Confirmation of the draft, or the reason it could not be created.
    """
    party = adults + kids
    nights = nights_between(check_in, check_out)
    if nights <= 0:
        return f"Invalid dates: check-out ({check_out}) must be after check-in ({check_in})."

    stay = full_collection.find_one({"_id": listing_id})
    if not stay:
        return f"No stay found with id={listing_id}. Use an id from suggest_stays."

    # ---- Validate availability and capacity ----
    if (stay.get("availability") or {}).get("availability_365", 0) <= 0:
        return f"'{stay.get('name')}' has no availability. Please choose another stay."

    if stay.get("accommodates", 0) < party:
        return (
            f"'{stay.get('name')}' sleeps {stay.get('accommodates')} but the party "
            f"is {party}. Please choose a larger stay."
        )

    # minimum_nights is stored as a STRING in this dataset- coerce before comparing
    <CODE_BLOCK_13>
    if nights < min_nights:
        return (
            f"'{stay.get('name')}' requires at least {min_nights} nights but the trip "
            f"is {nights}. Extend the stay or choose another place."
        )

    # ---- Build and write the draft ----
    price_per_night = float(stay["price"])
    stay_total = price_per_night * nights
    draft = {
        "_id": slugify(full_name, destination_city, check_in),
        "full_name": full_name,
        "draft": True,
        "party": {"adults": adults, "kids": kids},
        "destination": destination_city,
        "booking_dates": {"check_in": check_in, "check_out": check_out},
        "stay": {
            "listing_id": listing_id,
            "name": stay.get("name"),
            "price_per_night": price_per_night,
            "nights": nights,
            "stay_total": stay_total,
        },
        "flight": {
            "summary": flight_summary,
            "price_total": float(flight_price_total),
        },
        "estimated_total": stay_total + float(flight_price_total),
        "lastModified": datetime.now(timezone.utc).isoformat(),
    }

    # Idempotent: the same trip updates its draft instead of duplicating it
    <CODE_BLOCK_14>

    return (
        f"Draft booking '{draft['_id']}' created for {full_name}.\n"
        f"  Stay: {stay.get('name')} ({nights} nights, ${stay_total:g})\n"
        f"  Flight: {flight_summary or 'not selected'} (${float(flight_price_total):g})\n"
        f"  Party: {adults} adults, {kids} children\n"
        f"  Estimated total: ${draft['estimated_total']:g}"
    )

In [ ]:
# Collect the tools the agent can use
tools = [
    get_trip_requirements,
    check_existing_bookings,
    search_flights,
    suggest_stays,
    create_booking_draft,
]
tools_by_name = {t.name: t for t in tools}
print(list(tools_by_name))

# Step 6: Instantiate the LLM

In [ ]:
from utils import get_llm

# Obtain the Langchain LLM object using the `get_llm` function from the `utils` module
llm = get_llm(LLM_PROVIDER)
# Give the LLM access to the tools
<CODE_BLOCK_15>

In [ ]:
# The agent's instructions. Note the explicit ordering rules- the system prompt is
# where an agent's "process" lives.
SYSTEM_PROMPT = (
    "You are a helpful travel booking agent.\n"
    "Follow this process:\n"
    "1. Find out who is travelling: full name, number of adults, number of children,\n"
    "   the city they are departing from, the destination, and the travel dates.\n"
    "   Use get_trip_requirements to see what is still missing, then ASK the traveller\n"
    "   for anything you don't know. Ask for several details at once, not one at a time.\n"
    "2. Once you have everything, call check_existing_bookings. If an overlapping\n"
    "   booking already exists, tell the traveller and STOP.\n"
    "3. Call search_flights to find how they get there.\n"
    "4. Call suggest_stays and present the options clearly, with prices.\n"
    "5. Ask which stay and flight they want. NEVER call create_booking_draft until\n"
    "   the traveller has explicitly chosen and approved.\n"
    "Be concise. Do not invent flights, stays, or prices- only use tool results.\n"
    f"You have access to these tools: {', '.join(t.name for t in tools)}."
)

In [ ]:
# Check the LLM picks the right tool for a travel request
llm_with_tools.invoke(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "Find me flights from New York to Barcelona for 2 adults",
        },
    ]
).tool_calls

# Step 7: Define the graph state

The previous labs tracked only `messages`. A booking agent needs more: it has to know
**what it has collected** and **whether the traveller has approved**.

Tracking approval in typed state (rather than trusting the model to remember) is what
makes the confirmation step in Step 9 reliable.

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict


class TripRequest(TypedDict, total=False):
    """Trip details collected from the traveller."""

    full_name: str
    adults: int
    kids: int
    origin_city: str
    destination_city: str
    check_in: str
    check_out: str


class GraphState(TypedDict):
    """State carried through the graph."""

    <CODE_BLOCK_16>
    # What we know about the trip so far
    <CODE_BLOCK_17>
    # Set to True only after the traveller approves
    <CODE_BLOCK_18>

# Step 8: Define the graph nodes

In [ ]:
from langchain_core.messages import ToolMessage
from langgraph.types import interrupt


def agent(state: GraphState) -> Dict:
    """Agent node: decides what to say or which tool to call."""
    messages = state["messages"]
    result = llm_with_tools.invoke(
        [{"role": "system", "content": SYSTEM_PROMPT}, *messages]
    )
    return {"messages": [result]}

In [ ]:
def tool_node(state: GraphState) -> Dict:
    """Tool node: runs every tool the agent asked for."""
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        selected_tool = tools_by_name[tool_call["name"]]
        <CODE_BLOCK_19>
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

### The confirmation gate

This is the important part. `interrupt()` **suspends the graph** and saves its state.
Nothing runs until you resume it with the traveller's answer.

Because the write tool sits *behind* this node, the agent **cannot** create a booking
without a real human reply — it isn't relying on the model to behave. If the traveller
declines, the tool call is discarded and the agent is told so.

📚 https://docs.langchain.com/oss/python/langgraph/interrupts

In [ ]:
def confirm(state: GraphState) -> Dict:
    """Human-in-the-loop gate: pause for approval before writing a booking."""
    last_message = state["messages"][-1]
    # There is exactly one pending create_booking_draft call at this point
    booking_call = next(
        call
        for call in last_message.tool_calls
        if call["name"] == "create_booking_draft"
    )
    args = booking_call["args"]

    # Suspend the graph and surface the booking for review
    <CODE_BLOCK_20>

    approved = str(answer).strip().lower() in {"yes", "y", "approve", "approved", "ok"}
    if not approved:
        # Answer the pending tool call WITHOUT writing anything
        return {
            "approved": False,
            "messages": [
                ToolMessage(
                    content=(
                        "The traveller did NOT approve this booking. "
                        "No draft was created. Ask what they would like to change."
                    ),
                    tool_call_id=booking_call["id"],
                )
            ],
        }

    # Approved: now- and only now- run the write tool
    <CODE_BLOCK_21>
    return {
        "approved": True,
        "trip": {
            "full_name": args.get("full_name", ""),
            "destination_city": args.get("destination_city", ""),
            "check_in": args.get("check_in", ""),
            "check_out": args.get("check_out", ""),
            "adults": args.get("adults", 0),
            "kids": args.get("kids", 0),
        },
        "messages": [
            ToolMessage(content=observation, tool_call_id=booking_call["id"])
        ],
    }

# Step 9: Build the graph

The routing function is what makes the gate **structural** rather than advisory: any
attempt to call `create_booking_draft` is routed to `confirm`, never straight to `tools`.

```
START → agent ⇄ tools
          └── create_booking_draft? → confirm ⏸ → agent → END
```

In [ ]:
from langgraph.graph import StateGraph, START, END


def route_tools(state: GraphState) -> str:
    """Route to the confirmation gate for writes, the tool node otherwise."""
    messages = state.get("messages", [])
    if not messages:
        raise ValueError(f"No messages found in input state: {state}")

    ai_message = messages[-1]
    tool_calls = getattr(ai_message, "tool_calls", [])
    if not tool_calls:
        return END

    # A write always goes through the human-in-the-loop gate first
    <CODE_BLOCK_22>

In [ ]:
graph = StateGraph(GraphState)
graph.add_node("agent", agent)
graph.add_node("tools", tool_node)
<CODE_BLOCK_23>

graph.add_edge(START, "agent")
graph.add_edge("tools", "agent")
# After confirming (or declining) hand control back to the agent to reply
graph.add_edge("confirm", "agent")

<CODE_BLOCK_24>

`interrupt()` needs a **checkpointer** — that's what stores the suspended state so the
graph can resume later. Short-term memory isn't optional here; it's what makes
human-in-the-loop possible.

📚 https://docs.langchain.com/oss/python/langgraph/persistence#threads

In [ ]:
from langgraph.checkpoint.mongodb import MongoDBSaver

# Store conversation state in MongoDB
checkpointer = MongoDBSaver(mongodb_client)
<CODE_BLOCK_25>
app

# Step 10: Talk to the agent

Two helpers: one to send a message, one to answer a pending approval request.

The `thread_id` is what ties a conversation together — reuse it to continue the same
conversation, change it to start fresh.

In [ ]:
from langgraph.types import Command


def _render(step: Dict) -> None:
    """Print whatever the graph just produced."""
    # A pending approval request surfaces on the `__interrupt__` key
    if "__interrupt__" in step:
        payload = step["__interrupt__"][0].value
        print("\n" + "=" * 70)
        print("APPROVAL NEEDED:", payload["question"])
        for key, value in payload["booking"].items():
            print(f"    {key}: {value}")
        print("Reply with resume_graph(thread_id, 'yes') or 'no'.")
        print("=" * 70)
        return

    if "messages" not in step:
        return
    step["messages"][-1].pretty_print()


def execute_graph(thread_id: str, user_input: str) -> None:
    """Send a message to the agent and stream the response."""
    config = {"configurable": {"thread_id": thread_id}}
    for step in app.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        config,
        stream_mode="values",
    ):
        _render(step)


def resume_graph(thread_id: str, answer: str) -> None:
    """Answer a pending approval request and let the graph continue."""
    config = {"configurable": {"thread_id": thread_id}}
    <CODE_BLOCK_26>
        _render(step)

### Turn 1: the agent asks who is travelling

It has no details yet, so it should ask rather than guess.

In [ ]:
execute_graph("trip-1", "Hi, I'd like to plan a family holiday to Barcelona.")

### Turn 2: give it everything at once

Now it should check for duplicates, search flights, and suggest stays. Notice the stays
all sleep at least 4 and include child-friendly amenities.

In [ ]:
execute_graph(
    "trip-1",
    "I'm Grace Hopper, travelling with my partner and 2 kids, so 2 adults and "
    "2 children. We're flying from New York, 2026-04-10 to 2026-04-17. "
    "We'd like somewhere near the beach with a kitchen.",
)

### Turn 3: choose, and hit the approval gate

When the agent tries to write, the graph **suspends**. Nothing has been saved yet.

In [ ]:
execute_graph(
    "trip-1",
    "Let's go with the first flight and the first stay. Please create the draft.",
)

In [ ]:
# Proof that nothing was written while the graph is suspended
state = app.get_state({"configurable": {"thread_id": "trip-1"}})
print("graph is paused at:", state.next)
print("drafts in the database:", drafts_collection.count_documents({}))

### Turn 4a: decline

Say no and the booking is discarded — the agent is told and asks what to change.

In [ ]:
resume_graph("trip-1", "no")
print("\ndrafts after declining:", drafts_collection.count_documents({}))

### Turn 4b: approve

Ask again, then approve. Only now is the draft written.

In [ ]:
execute_graph("trip-1", "Actually yes, go ahead and create that booking.")

In [ ]:
resume_graph("trip-1", "yes")

In [ ]:
# The draft as stored in MongoDB
pprint(drafts_collection.find_one({"full_name": "Grace Hopper"}))

### Duplicate protection

Start a **new** conversation and ask for the same trip. The agent finds the existing
booking and stops instead of creating a second one.

In [ ]:
execute_graph(
    "trip-2",
    "I'm Grace Hopper. Book me a place in Barcelona from 2026-04-12 to 2026-04-19 "
    "for 2 adults and 2 kids, flying from New York.",
)

# 🦹‍♀️ Step 11: Remember the traveller between trips

Right now every conversation starts from nothing. A real travel agent remembers that
you fly from JFK, travel with two small children, and always need a crib.

`MongoDBStore` keeps those facts **outside** any single conversation and retrieves them
by vector search, so the agent stops asking questions it already knows the answer to.

📚 https://docs.langchain.com/oss/python/langgraph/add-memory#use-semantic-search

> **Two things to expect on the next cell:**
> - The first run creates a second vector index (for memories) and may raise a
>   `TimeoutError` after ~15s. The index is still being built — just **re-run the cell**
>   and it will succeed.
> - Newly saved memories take a few seconds to become searchable. If a recall comes back
>   empty, wait a moment and try again.

In [ ]:
from langgraph.store.mongodb import MongoDBStore, create_vector_index_config
from langchain_voyageai import VoyageAIEmbeddings
import uuid

memory_collection = mongodb_client[DB_NAME]["traveller_memories"]

# `voyage-4` again- the same family used for the stay embeddings
mongodb_store = MongoDBStore(
    collection=memory_collection,
    index_config=create_vector_index_config(
        embed=VoyageAIEmbeddings(model=QUERY_MODEL),
        dims=1024,
    ),
)

# All memories live under one traveller id for this lab
USER_ID = "user_1"

In [ ]:
@tool
def save_traveller_preference(preference: str) -> str:
    """
    Remember a travel preference or fact about the traveller for future trips,
    e.g. their name, home airport, party make-up, or what they like in a stay.

    Args:
        preference (str): The fact to remember.
    """
    mongodb_store.put(
        (USER_ID,),
        key=str(uuid.uuid4()),
        value={"text": preference},
    )
    return f"Noted: {preference}"

In [ ]:
# Add the memory tool and rebind
tools = [
    get_trip_requirements,
    check_existing_bookings,
    search_flights,
    suggest_stays,
    create_booking_draft,
    save_traveller_preference,
]
tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

The agent node now looks up relevant memories **before** replying, and uses them to
pre-fill what it already knows.

In [ ]:
def agent(state: GraphState) -> Dict:
    """Agent node, now with recall of past preferences."""
    messages = state["messages"]

    # Retrieve preferences relevant to what the traveller just said
    <CODE_BLOCK_27>
    memories = "\n".join(f"- {h.value['text']}" for h in hits)
    memories = memories or "- nothing remembered yet"

    system_prompt = (
        SYSTEM_PROMPT
        + "\nIf the traveller mentions a lasting preference (their name, home airport,"
        "\nwho they travel with, what they like in a stay), save it with"
        "\nsave_traveller_preference."
        "\nUse what you already remember instead of asking again. Confirm rather than"
        "\nre-ask, e.g. 'Flying from New York as usual?'."
        f"\n\nWhat you remember about this traveller:\n{memories}"
    )

    result = llm_with_tools.invoke(
        [{"role": "system", "content": system_prompt}, *messages]
    )
    return {"messages": [result]}

In [ ]:
# Rebuild the graph so it picks up the new agent function
graph = StateGraph(GraphState)
graph.add_node("agent", agent)
graph.add_node("tools", tool_node)
graph.add_node("confirm", confirm)
graph.add_edge(START, "agent")
graph.add_edge("tools", "agent")
graph.add_edge("confirm", "agent")
graph.add_conditional_edges(
    "agent",
    route_tools,
    {"tools": "tools", "confirm": "confirm", END: END},
)

# Short-term memory (checkpointer) AND long-term memory (store)
<CODE_BLOCK_28>

### Teach it something in one conversation

In [ ]:
execute_graph(
    "memory-1",
    "Remember for next time: I'm Grace Hopper, I always fly out of New York, "
    "I travel with 2 adults and 2 young children, and we need a crib and a kitchen.",
)

### Then start a brand new conversation

A different `thread_id` means no shared chat history. Anything the agent recalls here
came from **long-term memory** — it should already know the party size and home airport.

In [ ]:
execute_graph("memory-2", "I'd like to go to Porto in September. Can you help?")

In [ ]:
# What the agent has stored
for memory in mongodb_store.search((USER_ID,), query="travel preferences", limit=10):
    print("-", memory.value["text"])

---

## What you built

| Capability | How |
| --- | --- |
| Understands vague requests | vector search over 5,555 stays |
| Respects hard constraints | `filter` fields on the vector index |
| Finds connecting flights | `$graphLookup` over 66,985 routes |
| Fast graph traversal | indexing `connectToField` (~40x) |
| Avoids duplicate bookings | date-overlap query before writing |
| Never books without consent | `interrupt()` + a routed `confirm` node |
| Remembers you between trips | `MongoDBStore` with semantic recall |

### Things to try

- Ask for **Kauai to Istanbul** — no direct route, so `$graphLookup` finds a connection
- Ask for **Kauai to Porto** — genuinely unreachable, even with one stop
- Ask for a party of **12** and watch the capacity validation refuse it
- Approve a booking, then request the same trip again in a new thread
- Run `scripts/benchmark_graphlookup.py --drop` and redo Step 4a from scratch